# Final LightGBM Training and Kaggle Submission
This notebook trains the selected tuned LightGBM model on the complete IEEE-CIS training dataset.

The final model was selected using:

- A chronological train-validation split
- Temporal cross-validation
- ROC-AUC as the primary selection metric
- Average Precision as a secondary metric

The final model will generate fraud probabilities for the competition test dataset.

In [ ]:
# Data Proccessing 
import pandas as pd
from lightgbm import LGBMClassifier

#data_path = "../../Data/"

# Load the datasets
train_transaction = pd.read_csv("../../Data/train_transaction.csv")
train_identity = pd.read_csv("../../Data/train_identity.csv")
test_transaction = pd.read_csv("../../Data/test_transaction.csv")
test_identity = pd.read_csv("../../Data/test_identity.csv")
sample_submission = pd.read_csv( "../../Data/sample_submission.csv")

# the number of columns is matched with the Test data 
#Train transaction: (590540, 394)
#Train identity: (144233, 41)
#Test transaction: (506691, 393)
#Test identity: (141907, 41)
#Sample submission: (506691, 2)


Features match (Transaction): False
Features match (Idintity): True


In [14]:
# Check the column names in both dataset (Train + Test)

# Compare identity column names
print("Identity - Train only:",train_identity.columns.difference(test_identity.columns).tolist())
print("Identity - Test only:",test_identity.columns.difference(train_identity.columns).tolist())

# Compare transaction column names
print("Transaction - Train only:",train_transaction.columns.difference(test_transaction.columns).tolist())
print("Transaction - Test only:",test_transaction.columns.difference(train_transaction.columns).tolist())


Identity - Train only: ['id_01', 'id_02', 'id_03', 'id_04', 'id_05', 'id_06', 'id_07', 'id_08', 'id_09', 'id_10', 'id_11', 'id_12', 'id_13', 'id_14', 'id_15', 'id_16', 'id_17', 'id_18', 'id_19', 'id_20', 'id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_27', 'id_28', 'id_29', 'id_30', 'id_31', 'id_32', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38']
Identity - Test only: ['id-01', 'id-02', 'id-03', 'id-04', 'id-05', 'id-06', 'id-07', 'id-08', 'id-09', 'id-10', 'id-11', 'id-12', 'id-13', 'id-14', 'id-15', 'id-16', 'id-17', 'id-18', 'id-19', 'id-20', 'id-21', 'id-22', 'id-23', 'id-24', 'id-25', 'id-26', 'id-27', 'id-28', 'id-29', 'id-30', 'id-31', 'id-32', 'id-33', 'id-34', 'id-35', 'id-36', 'id-37', 'id-38']
Transaction - Train only: ['isFraud']
Transaction - Test only: []


In [ ]:
# Match test identity column names with train identity
test_identity.columns = test_identity.columns.str.replace( "-", "_",regex=False)

# Mark transactions that have identity information
train_identity["HasIdentity"] = 1
test_identity["HasIdentity"] = 1

# Merge transaction and identity datasets
train_merged = train_transaction.merge(
    train_identity,
    on="TransactionID",
    how="left"
)

test_merged = test_transaction.merge(
    test_identity,
    on="TransactionID",
    how="left"
)
# Complete the identity indicator
train_merged["HasIdentity"] = (train_merged["HasIdentity"].fillna(0).astype("int8"))
test_merged["HasIdentity"] = (test_merged["HasIdentity"].fillna(0).astype("int8"))

# Create the transaction hour feature
train_merged["TranHour"] = ((train_merged["TransactionDT"] // 3600) % 24).astype("int8")
test_merged["TranHour"] = ((test_merged["TransactionDT"] // 3600) % 24).astype("int8")



In [16]:
# Separate the target and remove non-model columns
y_full = train_merged["isFraud"].copy()

X_full = train_merged.drop(columns=["TransactionID", "isFraud", "TransactionDT"])
X_test = test_merged.drop(columns=["TransactionID", "TransactionDT"])

# Match test feature order with training features
X_test = X_test.reindex(columns=X_full.columns)

print("X_full:", X_full.shape)
print("y_full:", y_full.shape)
print("X_test:", X_test.shape)
print("Features match:", X_full.columns.equals(X_test.columns))

X_full: (590540, 433)
y_full: (590540,)
X_test: (506691, 433)
Features match: True


In [ ]:
# Identify categorical features
categorical_features = (X_full.select_dtypes(include=["object", "string"]).columns.tolist())

# Prepare categorical features for LightGBM
for column in categorical_features + ["TranHour"]:
    X_full[column] = X_full[column].astype("category")

    X_test[column] = pd.Categorical(
        X_test[column],
        categories=X_full[column].cat.categories
    )
    
print("Categorical features:", len(categorical_features) + 1)
print("Numerical features:", X_full.shape[1] - len(categorical_features) - 1)
print("Total features:", X_full.shape[1])
print("Features match:", X_full.columns.equals(X_test.columns))

# Final Model Training
The tuned LightGBM model is trained on the complete training dataset
used the following ..
* temporal cross-validation
* 379 boosting iterations

In [ ]:
# Create the final tuned LightGBM model
final_lgbm_model = LGBMClassifier(
    objective="binary",
    n_estimators=379,
    learning_rate=0.05,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=100,
    colsample_bytree=0.75,
    reg_alpha=0,
    reg_lambda=0,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

# Train on the complete training dataset
final_lgbm_model.fit(
    X_full,
    y_full
)

print("Final model training completed.")


Final model training completed.
Training rows: 590540
Number of features: 433
Number of trees: 379


# Test Predictions and Submission File
The final model generates a fraud probability for every test transaction

In [ ]:
# Predict fraud probabilities for the test dataset
test_probability = final_lgbm_model.predict_proba(X_test)[:, 1]

# Create the submission using the official sample format
submission = sample_submission[["TransactionID"]].copy()
submission["isFraud"] = test_probability

display(submission.head())

Transaction IDs match: True
Submission shape: (506691, 2)
Missing probabilities: 0
Unique Transaction IDs: True
All probabilities valid: True
Minimum probability: 4.350810645068795e-05
Maximum probability: 0.9995656430020181
Average probability: 0.03075621851632756


,TransactionID,isFraud
0,3663549,0.004646
1,3663550,0.002839
2,3663551,0.005051
3,3663552,0.001880
4,3663553,0.002055


In [ ]:
# Save the submission file
submission.to_csv("ieee_fraud_lightgbm_tuned_submissionddddd.csv",index=False)

# Save the final trained LightGBM model
final_lgbm_model.booster_.save_model("final_lightgbm_model.txt")


# Kaggle Late Submission Results

The final tuned LightGBM model was evaluated through a Kaggle late submission.

* Evaluation Dataset (ROC-AUC)
Local validation: [0.9181]
Kaggle private test subset: [0.909055] 
Kaggle public test subset: [0.932229] 

The similarity between the local chronological validation result and the Kaggle test results indicates that the validation strategy provided a reasonable estimate of future model performance.
